In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

import torch
from pytorch_tabnet.tab_model import TabNetClassifier

In [ ]:
'''
In this approach, we will combine all the features from both the omics datasets--we will concat the RNASeq dataset of LUAD+LUSC with CNV of LUAD+LUSC.
This will create a dataset that will have same number of features but samples would increase. The approach will be an early-fusion approach.
'''

rna_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_luad.csv'))
rna_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_lusc.csv'))

rna_luad_xena['label'] = 1
rna_lusc_xena['label'] = 0
df_rna_xena = pd.concat([rna_luad_xena, rna_lusc_xena], axis=0)

cnv_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_luad.csv'))
cnv_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_lusc.csv'))

cnv_luad_xena['label'] = 1
cnv_lusc_xena['label'] = 0
df_cnv_xena = pd.concat([cnv_luad_xena, cnv_lusc_xena], axis=0)

In [ ]:
## select only common features and intersect both dataframes

common_genes = df_rna_xena.columns.intersection(df_cnv_xena.columns)

In [ ]:
len(common_genes)

In [ ]:
_rna_ = df_rna_xena[df_rna_xena.columns.intersection(common_genes)]
_cnv_ = df_cnv_xena[df_cnv_xena.columns.intersection(common_genes)]

In [ ]:
df = pd.concat([_rna_, _cnv_])

In [ ]:
df.isnull().values.any()

In [ ]:
## perform classification using XGBoost
def perform_classification(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = XGBClassifier(device='cuda')

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = XGBClassifier(device='cuda')
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

    # Metrics (against validation predictions)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
## perform classification using MLP
def perform_classification_mlp(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = MLPClassifier(random_state=random_state)

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = MLPClassifier(random_state=random_state)
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

    # Metrics (against validation predictions)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
## perform classification using SVC
def perform_classification_svc(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = SVC(probability=True)

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = SVC(probability=True)
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

    # Metrics (against validation predictions)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
def perform_tabnet(df, random_state):
    """
    Check the performance of TabNet
    """
    
    accuracies = []
    aurocs = []
    conf_mats = []
    y_trues = []
    y_preds_proba = []

    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    
    for train_idx, test_idx in skfold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
    
        tabnet_model = TabNetClassifier(
            device_name = 'cuda',
            seed = random_state,
            verbose = 0
        )
        tabnet_model.fit(
            X_train, y_train,
            eval_set = [(X_test, y_test)],
            # max_epochs = 3,
            eval_metric = ['auc', 'accuracy'],
            batch_size = 512,
            patience = 0
        )
    
        y_pred_val = tabnet_model.predict(X_test)
        y_pred_prob = tabnet_model.predict_proba(X_test)[:,1]

        y_trues.append(y_test)
        y_preds_proba.append(y_pred_prob)
        
        # Compute metrics
        accuracy_val = accuracy_score(y_test, y_pred_val)
        auroc = roc_auc_score(y_test, y_pred_prob)
        cm = confusion_matrix(y_test, y_pred_val)
    
        # Store results
        accuracies.append(accuracy_val)
        
        aurocs.append(auroc)
        conf_mats.append(cm)

        del tabnet_model
        with torch.no_grad():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Compute mean metrics across folds
    mean_accuracy = np.mean(accuracies)
    
    mean_auroc = np.mean(aurocs)
    total_cm = np.sum(conf_mats, axis=0)  # Summing up all confusion matrices
    
    
    results = {
        'accuracy_val':mean_accuracy,
        'auroc':mean_auroc,
        'cm':total_cm,
        'y_proba':np.hstack(y_preds_proba),
        'y_true':np.hstack(y_trues),
    }
    
    return results

In [ ]:
def dump_all_results(results, fname, seed, model):
    
    # Extract confusion matrix values
    TN, FP, FN, TP = results['cm'].ravel()
    
    # Create results DataFrame
    results_df = pd.DataFrame({
        "Accuracy": [results['accuracy_val']],
        "Mean AUROC": [results['auroc']],
        "TN": [TN],
        "FP": [FP],
        "FN": [FN],
        "TP": [TP]
    })
    
    # results_df.to_csv(f"Z:/multiomics based manuscript/results_for_xena_rna_cnv_only/{fname}_{seed}.csv", index=False) ## Print results
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/JOINT_OMICS/{model}/{fname}_{seed}.csv", index=False) ## Print results

In [ ]:
for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df = df.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on joint dataframe
    results = perform_classification(_df, seed)

    print(f"JOINT_OMICS results for seed: {seed}:\t Acc : {results['accuracy_val']} \t AUROC: {results['auroc']}")
    
    dump_all_results(results, 'results_joint_omics', seed, 'XGB')
    print("--------------------------------------------------------------------------------")
    

In [ ]:
for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df = df.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on joint dataframe
    results = perform_classification_mlp(_df, seed)

    print(f"JOINT_OMICS results for seed: {seed}:\t Acc : {results['accuracy_val']} \t AUROC: {results['auroc']}")
    
    dump_all_results(results, 'results_joint_omics', seed, 'MLP')
    print("--------------------------------------------------------------------------------")
    

In [ ]:
for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df = df.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on joint dataframe
    results = perform_classification_svc(_df, seed)

    print(f"JOINT_OMICS results for seed: {seed}:\t Acc : {results['accuracy_val']} \t AUROC: {results['auroc']}")
    
    dump_all_results(results, 'results_joint_omics', seed, 'SVC')
    print("--------------------------------------------------------------------------------")
    

In [ ]:
for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df = df.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on joint dataframe
    results = perform_tabnet(_df, seed)

    print(f"JOINT_OMICS results for seed: {seed}:\t Acc : {results['accuracy_val']} \t AUROC: {results['auroc']}")
    
    dump_all_results(results, 'results_joint_omics', seed, 'TABNET')
    print("--------------------------------------------------------------------------------")
    